# Qwen3 0.6B Fine-Tuning — Spam Classification (2007/2008 Dataset)

Fine-tunes `Qwen/Qwen3-0.6B` on the combined TREC-2007 + CEAS-2008 spam dataset using LoRA and `Qwen3ForSequenceClassification` (binary classification head, no text generation).

In [1]:
import hashlib
import json
import os
from pathlib import Path

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import torch
from aim_tracking import (
    compute_binary_classification_metrics,
    create_aim_callbacks,
    summarize_text_classification_dataset,
)
from datasets import ClassLabel, DatasetDict, load_dataset
from dataset.combine import combine_datasets
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)


In [ ]:
MODEL_ID = "Qwen/Qwen3-0.6B"

SEED = 67
TRAIN_SPLIT = 0.999
VALIDATION_SPLIT = 0.0005
TEST_SPLIT = 0.0005
HOLDOUT_SPLIT = VALIDATION_SPLIT + TEST_SPLIT
MAX_SEQ_LENGTH = 448

# Tuned for an ~9 hour Apple Silicon run on an 18 GB MacBook Pro.
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
NUM_TRAIN_EPOCHS = 2
LEARNING_RATE = 8e-5
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 0.5

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
MPS_MEMORY_FRACTION = 0.92

AIM_EXPERIMENT_NAME = "qwen3-0.6b-sequence-class"
AIM_SYSTEM_TRACKING_INTERVAL = 10


In [3]:
def find_device() -> str:
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"

DEVICE = find_device()
CUDA = DEVICE == "cuda"
IS_MPS = DEVICE == "mps"

if IS_MPS:
    if hasattr(torch.mps, "empty_cache"):
        torch.mps.empty_cache()
    if hasattr(torch.mps, "set_per_process_memory_fraction"):
        torch.mps.set_per_process_memory_fraction(MPS_MEMORY_FRACTION)
    print("⚠️ Using MPS acceleration on Apple Silicon.")
    if hasattr(torch.mps, "recommended_max_memory"):
        print(f"Recommended max MPS working set: {torch.mps.recommended_max_memory() / 1024**3:.2f} GB")
    print(f"MPS per-process memory fraction: {MPS_MEMORY_FRACTION:.2f}")
elif CUDA:
    print(f"✅ CUDA is available. Found {torch.cuda.device_count()} device(s).")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
    print("Will use bitsandbytes quantization.")
else:
    print("❌ Neither MPS nor CUDA is available. Falling back to CPU, which will be very slow.")

set_seed(SEED)

⚠️ Using MPS acceleration on Apple Silicon.
Recommended max MPS working set: 13.32 GB
MPS per-process memory fraction: 0.92


## Dataset

Download and prepare the TREC-2007 + CEAS-2008 dataset before training. The combined parquet is cached and reused on later runs.


In [4]:
project_root = Path.cwd().resolve()
if not (project_root / "dataset").exists():
    project_root = project_root.parent

AIM_REPO_PATH = str(project_root)
DATASET_PATH = combine_datasets("training_all", combination_mode="mixed_50_50")

def sha256_file(path: str, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

DATASET_SHA256 = sha256_file(DATASET_PATH)

dataset = load_dataset("parquet", data_files=DATASET_PATH, split="train")
dataset = dataset.cast_column("label", ClassLabel(names=["valid", "spam"]))
dataset_summary = summarize_text_classification_dataset(dataset)
dataset_label_counts = dataset_summary["label_counts"]
dataset_source_counts = dataset_summary["source_counts"]
dataset_text_stats = dataset_summary["text_stats"]

print(f"Loaded {len(dataset)} rows from {DATASET_PATH}")
print(f"Dataset SHA256: {DATASET_SHA256}")
print(f"Columns: {dataset.column_names}")
print(f"Spam: {dataset_label_counts['spam']}, Ham: {dataset_label_counts['ham']}")
print(f"Source counts: {dataset_source_counts}")
print(f"Average subject chars: {dataset_text_stats['avg_subject_chars']:.2f}")
print(f"Average body chars: {dataset_text_stats['avg_body_chars']:.2f}")


Combined dataset already exists: /Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/dataset/combined_datasets/generated/ceas_2008__trec_2007__spam_0_5__133462257d.parquet
Loaded 83094 rows from /Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/dataset/combined_datasets/generated/ceas_2008__trec_2007__spam_0_5__133462257d.parquet
Dataset SHA256: 1be6714132a7d12f2c6bc35dcc31aa1cacfd5881cbf8dcc965c716f4838dcc98
Columns: ['subject', 'body', 'label', 'source']
Spam: 41547, Ham: 41547
Source counts: {'trec_2007': 49199, 'ceas_2008': 33895}
Average subject chars: 40.29
Average body chars: 2286.61


In [5]:
holdout = dataset.train_test_split(
    test_size=HOLDOUT_SPLIT,
    stratify_by_column="label",
    seed=SEED,
)
valid_test = holdout["test"].train_test_split(
    test_size=TEST_SPLIT / HOLDOUT_SPLIT,
    stratify_by_column="label",
    seed=SEED,
)

dataset = DatasetDict({
    "train": holdout["train"],
    "validation": valid_test["train"],
    "test": valid_test["test"],
})

for split in ["train", "validation", "test"]:
    labels = dataset[split]["label"]
    spam = labels.count(1)
    ham = labels.count(0)
    print(f"{split}: {len(dataset[split])} rows — spam: {spam}, ham: {ham}")

train: 83010 rows — spam: 41505, ham: 41505
validation: 42 rows — spam: 21, ham: 21
test: 42 rows — spam: 21, ham: 21


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def dataset_transform(sample):
    subject = (sample["subject"] or "").strip()
    body = (sample["body"] or "").strip()
    parts = []
    if subject:
        parts.append(f"Subject: {subject}")
    if body:
        parts.append(body)
    sample["text"] = "\n\n".join(parts).strip()
    return sample

dataset = dataset.map(dataset_transform, desc="Building email texts")
dataset = dataset.remove_columns(["subject", "body", "source"])
print(dataset)

Building email texts:   0%|          | 0/83010 [00:00<?, ? examples/s]

Building email texts:   0%|          | 0/42 [00:00<?, ? examples/s]

Building email texts:   0%|          | 0/42 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 83010
    })
    validation: Dataset({
        features: ['label', 'text'],
        num_rows: 42
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 42
    })
})


## Model & LoRA

In [7]:
if CUDA:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=2,
        quantization_config=bnb_config,
        dtype=torch.bfloat16,
    )
else:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=2,
        dtype=torch.float32,
    )
if hasattr(model, "score"):
    model.score = model.score.to(torch.float32)
model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id
model.config.problem_type = "single_label_classification"

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-0.6B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    modules_to_save=["score"],
)
model = get_peft_model(model, peft_config)
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
model.print_trainable_parameters()

trainable params: 10,094,592 || all params: 606,146,560 || trainable%: 1.6654


## Dataset Tokenization

Tokenize email text directly (no chat template) and rename `label` → `labels` for the Trainer.

In [ ]:
def tokenize(batch):
    encoded = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    encoded["length"] = [len(ids) for ids in encoded["input_ids"]]
    return encoded

tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing",
)

rows_before = sum(split.num_rows for split in tokenized_dataset.values())

tokenized_dataset = tokenized_dataset.filter(
    lambda x: x["length"] > 0,
    desc="Filtering out empty sequences"
)

# Count total rows across all splits after filtering
rows_after = sum(split.num_rows for split in tokenized_dataset.values())

print(f"Removed {rows_before - rows_after} empty samples!")


tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
print(tokenized_dataset)

Tokenizing:   0%|          | 0/83010 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/42 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/42 [00:00<?, ? examples/s]

Filtering out empty sequences:   0%|          | 0/83010 [00:00<?, ? examples/s]

Filtering out empty sequences:   0%|          | 0/42 [00:00<?, ? examples/s]

Filtering out empty sequences:   0%|          | 0/42 [00:00<?, ? examples/s]

Removed 0 empty samples!
DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask', 'length'],
        num_rows: 83010
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask', 'length'],
        num_rows: 42
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask', 'length'],
        num_rows: 42
    })
})


## Training

In [ ]:
if CUDA:
    print("✅ CUDA detected: using paged_adamw_8bit + bf16.")
    target_optim = "paged_adamw_8bit"
else:
    print("⚠️ CUDA not available: using adamw_torch.")
    target_optim = "adamw_torch"

effective_batch_size = TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
train_batches_per_epoch = int(np.ceil(len(tokenized_dataset["train"]) / TRAIN_BATCH_SIZE))
optimizer_steps_per_epoch = int(np.ceil(train_batches_per_epoch / GRADIENT_ACCUMULATION_STEPS))
eval_steps = max(1, optimizer_steps_per_epoch // 4)
print(f"Max sequence length: {MAX_SEQ_LENGTH}")
print(f"Effective batch size: {effective_batch_size}")
print(f"Optimizer steps per epoch: {optimizer_steps_per_epoch}")
print(f"Total optimizer steps: {optimizer_steps_per_epoch * NUM_TRAIN_EPOCHS}")
print(f"Evaluation/save cadence: every {eval_steps} optimizer steps")

training_args = TrainingArguments(
    report_to=["tensorboard"],
    run_name=AIM_EXPERIMENT_NAME,
    output_dir="./results/qwen3_0.6b_spam_mps",
    logging_dir=f"./runs/{AIM_EXPERIMENT_NAME}",
    logging_strategy="steps",
    logging_steps=1,
    logging_first_step=True,
    save_total_limit=3,
    seed=SEED,
    data_seed=SEED,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    lr_scheduler_type="cosine",

    optim=target_optim,
    max_steps=-1,

    bf16=CUDA,
    fp16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=eval_steps,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    # group_by_length=True,
    length_column_name="length",

    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    remove_unused_columns=False,
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


⚠️ CUDA not available: using adamw_torch.
Max sequence length: 448
Effective batch size: 16
Optimizer steps per epoch: 5189
Total optimizer steps: 10378
Evaluation/save cadence: every 1297 optimizer steps


In [11]:
def compute_metrics(eval_pred):
    metrics = compute_binary_classification_metrics(eval_pred)
    print(metrics)
    return metrics

run_config = {
    "model_id": MODEL_ID,
    "seed": SEED,
    "max_seq_length": MAX_SEQ_LENGTH,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "effective_batch_size": effective_batch_size,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "max_grad_norm": MAX_GRAD_NORM,
    "optim": training_args.optim,
    "device": DEVICE,
    "bf16": bool(training_args.bf16),
    "fp16": bool(training_args.fp16),
    "gradient_checkpointing": bool(training_args.gradient_checkpointing),
    "eval_steps": training_args.eval_steps,
    "save_steps": training_args.save_steps,
    "logging_steps": training_args.logging_steps,
    "tensorboard_log_dir": training_args.logging_dir,
    "output_dir": training_args.output_dir,
}

dataset_metadata = {
    "path": DATASET_PATH,
    "sha256": DATASET_SHA256,
    "rows": len(dataset),
    "spam": dataset_label_counts["spam"],
    "ham": dataset_label_counts["ham"],
    "sources": dataset_source_counts,
    "avg_subject_chars": dataset_text_stats["avg_subject_chars"],
    "avg_body_chars": dataset_text_stats["avg_body_chars"],
    "train_rows": len(tokenized_dataset["train"]),
    "validation_rows": len(tokenized_dataset["validation"]),
    "test_rows": len(tokenized_dataset["test"]),
}

lora_metadata = {
    "r": LORA_R,
    "alpha": LORA_ALPHA,
    "dropout": LORA_DROPOUT,
    "target_modules": [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    "modules_to_save": ["score"],
}

aim_callback, notebook_aim_callback = create_aim_callbacks(
    repo_path=AIM_REPO_PATH,
    experiment_name=AIM_EXPERIMENT_NAME,
    system_tracking_interval=AIM_SYSTEM_TRACKING_INTERVAL,
    run_config=run_config,
    dataset_metadata=dataset_metadata,
    lora_metadata=lora_metadata,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[aim_callback, notebook_aim_callback],
)


/Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/.venv/lib/python3.12/site-packages/aim/ext/utils.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [12]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1,Ham Precision,Ham Recall,Ham F1,Specificity,Balanced Accuracy,Matthews Corrcoef,Roc Auc,Pr Auc,Cross Entropy,Brier Score,Mean Abs Probability Error,Prediction Entropy,Spam Prediction Rate,Ham Prediction Rate,Mean Spam Probability,Mean Spam Probability On Spam,Mean Spam Probability On Ham,False Positive Count,False Negative Count,True Positive Count,True Negative Count
1,4.138419,1.660795,0.523810,0.513514,0.904762,0.655172,0.556757,0.523810,0.442971,0.556757,0.523810,0.442971,0.600000,0.142857,0.230769,0.142857,0.523810,0.073521,0.417234,0.438492,1.660795,0.389361,0.514296,0.383779,0.880952,0.119048,0.765328,0.751032,0.779625,18.000000,2.000000,19.000000,3.000000
2,2.235096,1.660158,0.523810,0.513514,0.904762,0.655172,0.556757,0.523810,0.442971,0.556757,0.523810,0.442971,0.600000,0.142857,0.230769,0.142857,0.523810,0.073521,0.417234,0.438492,1.660157,0.389335,0.514283,0.383817,0.880952,0.119048,0.765288,0.751006,0.779571,18.000000,2.000000,19.000000,3.000000
3,3.195742,1.659001,0.523810,0.513514,0.904762,0.655172,0.556757,0.523810,0.442971,0.556757,0.523810,0.442971,0.600000,0.142857,0.230769,0.142857,0.523810,0.073521,0.417234,0.438492,1.659001,0.389265,0.514256,0.383936,0.880952,0.119048,0.765146,0.750889,0.779402,18.000000,2.000000,19.000000,3.000000
4,2.996004,1.657109,0.523810,0.513514,0.904762,0.655172,0.556757,0.523810,0.442971,0.556757,0.523810,0.442971,0.600000,0.142857,0.230769,0.142857,0.523810,0.073521,0.417234,0.438492,1.657109,0.389178,0.514220,0.384082,0.880952,0.119048,0.764976,0.750757,0.779196,18.000000,2.000000,19.000000,3.000000
5,3.907072,1.654748,0.523810,0.513514,0.904762,0.655172,0.556757,0.523810,0.442971,0.556757,0.523810,0.442971,0.600000,0.142857,0.230769,0.142857,0.523810,0.073521,0.417234,0.438492,1.654748,0.389058,0.514190,0.384333,0.880952,0.119048,0.764678,0.750488,0.778867,18.000000,2.000000,19.000000,3.000000


{'accuracy': 0.5238095238095238, 'precision': 0.5135135135135135, 'recall': 0.9047619047619048, 'f1': 0.6551724137931034, 'macro_precision': 0.5567567567567567, 'macro_recall': 0.5238095238095238, 'macro_f1': 0.44297082228116713, 'weighted_precision': 0.5567567567567567, 'weighted_recall': 0.5238095238095238, 'weighted_f1': 0.4429708222811672, 'ham_precision': 0.6, 'ham_recall': 0.14285714285714285, 'ham_f1': 0.23076923076923078, 'specificity': 0.14285714285714285, 'balanced_accuracy': 0.5238095238095238, 'matthews_corrcoef': 0.07352146220938077, 'roc_auc': 0.4172335600907029, 'pr_auc': 0.4384918754959889, 'cross_entropy': 1.6607950411802732, 'brier_score': 0.38936106590372915, 'mean_abs_probability_error': 0.5142961045106252, 'prediction_entropy': 0.3837791681289673, 'spam_prediction_rate': 0.8809523809523809, 'ham_prediction_rate': 0.11904761904761904, 'mean_spam_probability': 0.7653284668922424, 'mean_spam_probability_on_spam': 0.7510323524475098, 'mean_spam_probability_on_ham': 0.7

KeyboardInterrupt: 

## Inference

In [ ]:
import datetime

# Generates the timestamp in HHmmDDMMYYYY format (e.g., 105014042026)
timestamp = datetime.datetime.now().strftime("%H%M%d%m%Y")
save_path = f"./results/qwen3_0.6b_spam_saved_weights_{timestamp}"

# Save the model and the tokenizer to the new timestamped directory
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
aim_callback.experiment["final_saved_model_path"] = save_path
aim_callback.experiment["saved_tokenizer_path"] = save_path

print(f"✅ Model safely saved to disk at: {save_path}")


✅ Model safely saved to disk at: ./results/qwen3_0.6b_spam_saved_weights_105114042026


In [ ]:
def _label_to_str(pred: int) -> str:
    return "spam" if pred == 1 else "valid"


def run_mail_classification(email_text: str) -> str:
    """Classify a raw email string as 'spam' or 'valid'."""
    email_text = email_text.strip()

    print(email_text)
    
    model.eval()
    model_device = next(model.parameters()).device
    inputs = tokenizer(
        email_text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        return_tensors="pt"
    ).to(model_device)
    with torch.no_grad():
        outputs = model(**inputs)
    pred = torch.argmax(outputs.logits, dim=-1).item()
    return _label_to_str(pred)

def run_mail_from_dataset_classification(dataset, index: int) -> str:
    """Classify a sample from the dataset by its text field."""
    return run_mail_classification(dataset[index]["text"])

## Evaluation

In [ ]:
trainer.args.per_device_eval_batch_size = 1

test_predictions = trainer.predict(tokenized_dataset["test"])
test_pred_labels = np.argmax(test_predictions.predictions, axis=-1)
test_true_labels = np.array(tokenized_dataset["test"]["labels"])

rounded_metrics = {
    key: round(value, 4)
    for key, value in test_predictions.metrics.items()
    if isinstance(value, (int, float))
}
print(rounded_metrics)

test_dataset = dataset["test"]
incorrect_samples = []
for i, (pred, actual) in enumerate(zip(test_pred_labels, test_true_labels)):
    if pred != actual:
        incorrect_samples.append({
            "index": i,
            "content": test_dataset[i]["text"],
            "actual": _label_to_str(int(actual)),
            "output": _label_to_str(int(pred)),
        })


print(f"Test accuracy: {(test_pred_labels == test_true_labels).mean():.4f}")
print(f"Mistakes: {len(incorrect_samples)} / {len(test_true_labels)}")

{'accuracy': 0.988, 'precision': 0.9790575916230366, 'recall': 0.9973333333333333, 'f1': 0.9881109643328929}
{'test_loss': 0.0328, 'test_accuracy': 0.988, 'test_precision': 0.9791, 'test_recall': 0.9973, 'test_f1': 0.9881, 'test_runtime': 109.9434, 'test_samples_per_second': 6.822, 'test_steps_per_second': 6.822}
Test accuracy: 0.4680
Mistakes: 399 / 750


In [ ]:
trainer.args.group_by_length = False

# Now run predictions (the output will perfectly map 1:1 with your original dataset)
test_predictions = trainer.predict(tokenized_dataset["test"])
test_pred_labels = np.argmax(test_predictions.predictions, axis=-1)

# Now it is safe to pull the true labels straight from the dataset
test_true_labels = np.array(tokenized_dataset["test"]["labels"])

rounded_metrics = {
    key: round(value, 4)
    for key, value in test_predictions.metrics.items()
    if isinstance(value, (int, float))
}
print(rounded_metrics)

test_dataset = dataset["test"]
incorrect_samples = []
for i, (pred, actual) in enumerate(zip(test_pred_labels, test_true_labels)):
    if pred != actual:
        incorrect_samples.append({
            "index": i,
            "content": test_dataset[i]["text"],
            "actual": _label_to_str(int(actual)),
            "output": _label_to_str(int(pred)),
        })

print(f"Test accuracy: {(test_pred_labels == test_true_labels).mean():.4f}")
print(f"Mistakes: {len(incorrect_samples)} / {len(test_true_labels)}")



{'accuracy': 0.988, 'precision': 0.9790575916230366, 'recall': 0.9973333333333333, 'f1': 0.9881109643328929}
{'test_loss': 0.0328, 'test_accuracy': 0.988, 'test_precision': 0.9791, 'test_recall': 0.9973, 'test_f1': 0.9881, 'test_runtime': 104.7232, 'test_samples_per_second': 7.162, 'test_steps_per_second': 7.162}
Test accuracy: 0.9880
Mistakes: 9 / 750


In [ ]:
print(incorrect_samples

In [ ]:
print(run_mail_classification("""\
Subject: E-mail details of the client.
Hi Greg, I have received the following contact info from the apache guys: "dan@apache.com",
I just wanted to check if this information is correct.
Best regards, John
"""))

Subject: E-mail details of the client.
Hi Greg, I have received the following contact info from the apache guys: "dan@apache.com",
I just wanted to check if this information is correct.
Best regards, John
valid


In [ ]:
print(run_mail_classification("""\
Subject: Free iPhone.
Hi Greg, You have won a free iPhone. Press the following link to receive your reward: "http://free-iphone.com"
"""))

Subject: Free iPhone.
Hi Greg, You have won a free iPhone. Press the following link to receive your reward: "http://free-iphone.com"
spam


In [ ]:
print(run_mail_classification("""
Subject: Wojciech, your virtual card is ready for use.
Your Revolut virtual card is ready for you. Head over to the ‘Cards’ tab in the app to access your new card details and start making payments online.
"""))

Subject: Wojciech, your virtual card is ready for use.
Your Revolut virtual card is ready for you. Head over to the ‘Cards’ tab in the app to access your new card details and start making payments online.
spam


#### print(run_mail_classification("""\
Subject: Obsługa języka polskiego.
Kup najnowszy ajfon za prawie darmo niskie ceny loteria, jesteś tysięcznym użytkownikiem!
Pozdrawiam, Wojciech
"""))

In [ ]:
print("dupa")

dupa


In [ ]:

print(run_mail_classification("""How are you doing? Can we meet later this afternoon? I just wanted to check if this information is correct."""))

spam
